# Real Estate Feasibility Engine – Python App
## Step 8: Stress testing + Monte Carlo

**What this notebook adds**
* `engine/stress.py` – four levers on top of the active scenario (**price, construction cost, sales speed, delay**), scenario comparison,
  one-way sensitivities, two-way heat maps, a tornado chart, **headroom** ("how far can prices fall before IRR hits the hurdle?") and **Monte Carlo**.
* `stress_view.py` – the new app screen **S · Stress & Monte Carlo** (live sliders, charts, probability results).
* 16 new automatic checks (total **165**), including a line-by-line comparison with the Excel STRESS sheet.

Run cells top to bottom with **Shift + Enter**. Steps 1–7 must have been run once.

### Cell 1 – Open the project folder

In [ ]:
import os, subprocess, sys, time, urllib.request
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
assert os.path.exists('dashboard.py'), 'Please run Step 7 first.'
try:
    import streamlit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'])
print('Ready in', BASE)

### Cell 2 – The stress & Monte Carlo engine (`engine/stress.py`)
**Levers** (same maths as the Excel STRESS levers): price −10% means every sales rate is 10% lower; cost +10% means construction rates
10% higher; sales period 1.5x means every product takes 50% longer to sell; delay +2 pushes every fixed launch and construction start
2 quarters later (linked products follow automatically).

**Monte Carlo:** each run draws a random price shock, cost shock and sales speed from a *triangular* range (lowest · most likely ·
highest) and a random delay from the probabilities you set, runs the full model, and records IRR, NPV, PAT and peak investment.
After 1,000 runs we can say things like *"there is an 85% chance developer IRR is at least 18%"*.

In [ ]:
%%writefile engine/stress.py
"""
engine/stress.py  -  STEP 8: stress testing and Monte Carlo (Excel STRESS sheet, and beyond).

Four levers act ON TOP of the active scenario (exactly like the Excel STRESS levers):
    price  : sales price shock      (-0.10 = all prices 10% lower)
    cost   : construction cost shock (+0.10 = construction 10% dearer)
    speed  : sales period multiplier (1.5 = every product takes 50% longer to sell)
    delay  : extra quarters added to every FIXED launch / construction start (linked products inherit it)

Functions
    stressed(project, ...)            -> a stressed COPY of the project (the original is never changed)
    outcome(project, ...)             -> headline results of one stressed run
    scenario_table / one_way / two_way / tornado / headroom
    monte_carlo(project, ranges, n)   -> n random combinations of the four levers -> probabilities
"""
from __future__ import annotations

import copy
from dataclasses import dataclass, field

import numpy as np

from .inputs import Project, Scenario
from .cashflow import run_model

LEVERS = {"price": "Sales price shock", "cost": "Construction cost shock", "speed": "Sales period multiplier", "delay": "Extra delay (quarters)"}
NEUTRAL = dict(price=0.0, cost=0.0, speed=1.0, delay=0)


def stressed(project: Project, price=0.0, cost=0.0, speed=1.0, delay=0, scenario: str | None = None) -> Project:
    """Copy of the project whose active scenario is combined with the levers (same maths as Excel)."""
    p = copy.deepcopy(project)
    if scenario:
        p.info.scenario = scenario
    s = p.scenario()
    combined = Scenario(f"{s.name} (stressed)", s.sales_period_mult * speed, (1 + s.price_change) * (1 + price) - 1,
                        (1 + s.cost_change) * (1 + cost) - 1, int(s.delay_q + delay))
    p.scenarios = [x for x in p.scenarios if x.name != combined.name] + [combined]
    p.info.scenario = combined.name
    return p


def outcome(project: Project, **levers) -> dict:
    r = run_model(stressed(project, **levers))
    m, pnl = r["cashflow"].metrics, r["cashflow"].pnl
    return dict(gdv=pnl["revenue"], total_cost=pnl["total_cost"], ebitda=pnl["ebitda"], pat=pnl["pat"],
                pat_margin=pnl["pat_margin"], developer_irr=m["developer_irr"], project_irr=m["project_irr"],
                developer_npv=m["developer_npv"], peak=m["peak_investment"], payback_q=m["payback_q"],
                equity_multiple=m["equity_multiple"], ok=not r["problems"], problems=r["problems"],
                developer_cf=r["cashflow"].developer_cf)


def scenario_table(project: Project) -> list:
    return [dict(scenario=s.name, **{k: v for k, v in outcome(project, scenario=s.name).items() if k != "developer_cf"})
            for s in project.scenarios]


DEFAULT_STEPS = {"price": [-0.15, -0.10, -0.05, 0.0, 0.05, 0.10, 0.15], "cost": [-0.05, 0.0, 0.05, 0.10, 0.15, 0.20],
                 "speed": [0.75, 1.0, 1.25, 1.5, 2.0], "delay": [0, 1, 2, 3, 4, 6]}


def one_way(project: Project, lever: str, values=None) -> list:
    return [dict(value=v, **{k: x for k, x in outcome(project, **{lever: v}).items() if k != "developer_cf"})
            for v in (values or DEFAULT_STEPS[lever])]


def two_way(project: Project, row_lever: str, row_values, col_lever: str, col_values, metric: str) -> np.ndarray:
    grid = np.full((len(row_values), len(col_values)), np.nan)
    for i, rv in enumerate(row_values):
        for j, cv_ in enumerate(col_values):
            v = outcome(project, **{row_lever: rv, col_lever: cv_})[metric]
            grid[i, j] = np.nan if v is None else v
    return grid


def tornado(project: Project, metric="developer_irr", shocks=None) -> list:
    """Downside and upside of each lever on one metric, sorted by size of the downside."""
    shocks = shocks or {"price": (-0.10, 0.10), "cost": (0.10, -0.05), "speed": (1.5, 0.75), "delay": (2, 0)}
    base = outcome(project)[metric]
    rows = []
    for lever, (down, up) in shocks.items():
        d, u = outcome(project, **{lever: down})[metric], outcome(project, **{lever: up})[metric]
        rows.append(dict(lever=LEVERS[lever], down_value=down, up_value=up, base=base, down=d, up=u,
                         impact=(d - base) if (d is not None and base is not None) else float("nan")))
    return sorted(rows, key=lambda x: x["impact"])


def _bisect(f, lo, hi, iters=60):
    """Root of a function that changes sign between lo and hi (None if it does not)."""
    flo, fhi = f(lo), f(hi)
    if flo is None or fhi is None or flo * fhi > 0:
        return None
    for _ in range(iters):
        mid = (lo + hi) / 2
        fm = f(mid)
        if fm is None: return None
        if flo * fm <= 0: hi, fhi = mid, fm
        else: lo, flo = mid, fm
    return (lo + hi) / 2


def headroom(project: Project) -> dict:
    """How far each lever can move before profit disappears or IRR falls to the hurdle."""
    hurdle = project.finance.benchmarks["developer_irr"]
    base_e = outcome(project)["ebitda"]
    e_p = outcome(project, price=-0.15)["ebitda"]
    e_c = outcome(project, cost=0.20)["ebitda"]
    irr_gap = lambda **k: (lambda v: None if v is None else v - hurdle)(outcome(project, **k)["developer_irr"])
    def irr_ok(**k):
        o = outcome(project, **k)
        return o["ok"], o["developer_irr"]
    delay_q = None
    for d in range(0, 13):                                   # first delay (in quarters) that pushes IRR below the hurdle
        ok, v = irr_ok(delay=d)
        if not ok: break
        if v is None or v < hurdle: delay_q = d; break
    speed_gap = lambda x: (lambda r: None if (not r[0] or r[1] is None) else r[1] - hurdle)(irr_ok(speed=x))
    return dict(
        price_to_zero_ebitda=-base_e / ((base_e - e_p) / 0.15) if base_e != e_p else None,      # exact: EBITDA is linear in price
        cost_to_zero_ebitda=base_e / ((base_e - e_c) / 0.20) if base_e != e_c else None,        # exact: linear in construction cost
        price_to_hurdle=_bisect(lambda x: irr_gap(price=x), -0.9, 0.0),
        cost_to_hurdle=_bisect(lambda x: irr_gap(cost=x), 0.0, 5.0),
        speed_to_hurdle=_bisect(speed_gap, 1.0, 3.0),
        delay_to_hurdle=delay_q, hurdle=hurdle)


# ------------------------------------------------------------------------------------- Monte Carlo
@dataclass
class McRanges:
    """Triangular ranges (lowest, most likely, highest) for each lever + probabilities for delay."""
    price: tuple = (-0.15, 0.0, 0.08)
    cost: tuple = (-0.05, 0.03, 0.20)
    speed: tuple = (0.8, 1.0, 1.8)
    delay_probs: dict = field(default_factory=lambda: {0: 0.50, 1: 0.25, 2: 0.15, 3: 0.10})


def monte_carlo(project: Project, ranges: McRanges | None = None, n: int = 1000, seed: int = 42, progress=None) -> dict:
    ranges = ranges or McRanges()
    rng = np.random.default_rng(seed)
    tri = lambda t: rng.triangular(t[0], t[1], t[2], n) if t[0] < t[2] else np.full(n, t[1])
    price, cost, speed = tri(ranges.price), tri(ranges.cost), tri(ranges.speed)
    d_vals, d_p = np.array(list(ranges.delay_probs)), np.array(list(ranges.delay_probs.values()), dtype=float)
    delay = rng.choice(d_vals, size=n, p=d_p / d_p.sum())
    irr, npv_, peak, pat = (np.full(n, np.nan) for _ in range(4))
    failed = 0
    for i in range(n):
        o = outcome(project, price=float(price[i]), cost=float(cost[i]), speed=float(speed[i]), delay=int(delay[i]))
        if not o["ok"]:
            failed += 1
        irr[i] = np.nan if o["developer_irr"] is None else o["developer_irr"]
        npv_[i], peak[i], pat[i] = o["developer_npv"], o["peak"], o["pat"]
        if progress and i % max(1, n // 50) == 0:
            progress(i / n)
    hurdle = project.finance.benchmarks["developer_irr"]
    valid = ~np.isnan(irr)
    pct = lambda a, q: float(np.nanpercentile(a, q))
    return dict(
        n=n, seed=seed, failed_runs=failed, samples=dict(price=price, cost=cost, speed=speed, delay=delay),
        irr=irr, npv=npv_, peak=peak, pat=pat, hurdle=hurdle,
        p_irr_above_hurdle=float(np.mean(irr[valid] >= hurdle)) if valid.any() else 0.0,
        p_npv_positive=float(np.mean(npv_ > 0)), p_loss=float(np.mean(pat < 0)),
        irr_p10=pct(irr, 10), irr_p50=pct(irr, 50), irr_p90=pct(irr, 90),
        npv_p10=pct(npv_, 10), npv_p50=pct(npv_, 50), npv_p90=pct(npv_, 90),
        peak_p50=pct(peak, 50), peak_p90=pct(peak, 90), peak_max=float(np.nanmax(peak)),
        pat_p10=pct(pat, 10), pat_p50=pct(pat, 50), pat_p90=pct(pat, 90))


### Cell 3 – Excel comparison helper (adds the STRESS sheet reader)

In [ ]:
%%writefile engine/excel_check.py
"""
engine/excel_check.py - reads the Excel FINAL model and compares it with the Python engine.
Works by LABELS (not cell addresses), so it is robust to row positions.
"""
from __future__ import annotations
import numpy as np
import openpyxl

NQ = 40
QCOL0 = 4      # quarter columns start at column D in TIMELINE / CURVES


def _row_values(ws, r):
    return np.array([ws.cell(r, c).value or 0 for c in range(QCOL0, QCOL0 + NQ)], dtype=float)


def read_excel_curves(path: str) -> dict:
    """Returns {'sales': {product: arr}, 'cons': {...}, 'cost': {cost name: arr}, 'profiles': {...}}"""
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    ws = wb["CURVES"]
    out = {"sales": {}, "cons": {}, "cost": {}, "cost_order": [], "profiles": {}}
    block, section = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  PRODUCT"):   section = "products"; continue
        if label.startswith("C.  AGGREGATE"): section = "profiles"; continue
        if label.startswith("D.  COST HEAD"): section = "costs"; continue
        if label.startswith("E.  CURVE CHECKS"): break
        if section == "products":
            if label.startswith("   Sales % (curve"):         out["sales"][block] = _row_values(ws, r)
            elif label.startswith("   Construction % (curve"): out["cons"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                   block = label
        elif section == "profiles":
            out["profiles"][label] = _row_values(ws, r)
        elif section == "costs":
            if label.startswith("   Spend %"): out["cost"][block] = _row_values(ws, r); out["cost_order"].append(block)
            elif not label.startswith("   "):   block = label
    return out


def read_excel_timeline(path: str) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["INPUTS"]
    prods = {}
    for r in range(1, ws.max_row + 1):          # product table: active rows have quarters in P..S
        if ws.cell(r, 5).value == "Yes" and isinstance(ws.cell(r, 16).value, (int, float)) and isinstance(ws.cell(r, 2).value, str):
            prods[ws.cell(r, 2).value] = tuple(int(ws.cell(r, c).value) for c in (16, 17, 18, 19))
    return prods


def read_excel_collections(path: str) -> dict:
    """Bookings, collections, totals, JV split and milestone due quarters from the COLLECTIONS sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["COLLECTIONS"]
    out = {"bookings": {}, "collections": {}, "triggers": {}, "total": None, "landowner": None, "developer": None}
    section, block = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("A.  MILESTONE"): section = "A"; continue
        if label.startswith("B.  PRODUCT"):   section = "B"; continue
        if label.startswith("C.  PROJECT"):   section = "C"; continue
        if label.startswith("D.  COLLECTION"): break
        if section == "A":
            if "   →   " in label:
                block = label.split("   →   ")[0]; out["triggers"][block] = []
            elif block and label.strip() and not label.startswith("   ") and ws.cell(r, 5).value:
                due = ws.cell(r, 6).value
                out["triggers"][block].append((label, int(due) if isinstance(due, (int, float)) else None))
        elif section == "B":
            if label.startswith("   Bookings (sales"):        out["bookings"][block] = _row_values(ws, r)
            elif label.startswith("   Collections (cash"):    out["collections"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                 block = label
        elif section == "C":
            if label.startswith("Total collections (cash"):   out["total"] = _row_values(ws, r)
            elif label.startswith("   Landowner share"):      out["landowner"] = _row_values(ws, r)
            elif label.startswith("   Developer share"):      out["developer"] = _row_values(ws, r)
    return out


def read_excel_cashflow(path: str) -> dict:
    """Quarterly cash rows from the CASHFLOW sheet and P&L / key metrics from the OUTPUT sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["CASHFLOW"]
    out = {"outflows": [], "rows": {}, "pnl": {}, "costs": [], "metrics": {}}
    want = {"Customer collections (100%)": "inflow", "Total outflows (excl. landowner share)": "total_outflow",
            "Cash paid to landowner this quarter": "lo_paid", "Developer net cash flow (pre-tax)": "developer_cf",
            "Developer net cash flow (post-tax, indicative)": "developer_cf_post_tax"}
    section = None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  CASH OUTFLOWS"): section = "B"; continue
        if label.startswith("C.  LANDOWNER"): section = None
        if label.startswith("Project net cash flow"): out["rows"]["project_cf"] = _row_values(ws, r)
        if section == "B" and not label.startswith("Total outflows"):
            out["outflows"].append(_row_values(ws, r))
        if label in want: out["rows"][want[label]] = _row_values(ws, r)
        if label.startswith("Total outflows"): section = None
    ws = wb["OUTPUT"]
    section = None
    for r in range(1, ws.max_row + 1):
        title = ws.cell(r, 2).value
        if isinstance(title, str):
            if title.startswith("B.  COST STATEMENT"): section = "B"
            elif title.startswith("C.  PROFIT"): section = "C"
            elif title.startswith("D.  JV"): section = "D"
            elif title.startswith("E.  KEY"): section = "E"
            continue
        label, val = ws.cell(r, 3).value, ws.cell(r, 4).value
        if not isinstance(label, str):
            continue
        if section == "B" and label not in ("Cost head",) and not label.startswith("TOTAL") and not label.startswith("Total development"):
            out["costs"].append(val or 0.0)
        if label == "SUM TOTAL REVENUE (A)": out["pnl"]["revenue"] = val
        if label == "TOTAL COST (B)": out["pnl"]["total_cost"] = val
        if label == "EBITDA (A − B)": out["pnl"]["ebitda"] = val
        if label == "Tax": out["pnl"]["tax"] = val
        if label == "PROFIT AFTER TAX (PAT)": out["pnl"]["pat"] = val
        if section == "E" and label[:3].strip().rstrip(".").isdigit():
            out["metrics"][label] = val
    return out


def read_excel_stress(path: str) -> dict:
    """Scenario comparison, one-way tables (C1-C4), heat maps (D1-D3) and headroom from the STRESS sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["STRESS"]
    out, section, mode, rows = {}, None, None, []
    val = lambda r, c: ws.cell(r, c).value
    r = 1
    while r <= ws.max_row:
        b = val(r, 2)
        if isinstance(b, str) and b[:3] in ("B. ", "C1.", "C2.", "C3.", "C4.", "D1.", "D2.", "D3.", "E. "):
            section = b.split(".")[0]
            if section in ("D1", "D2", "D3"):
                hdr = r + 2
                cols = [val(hdr, c) for c in range(3, 13) if isinstance(val(hdr, c), (int, float))]
                rr, grid, rowv = hdr + 1, [], []
                while isinstance(val(rr, 2), (int, float)):
                    rowv.append(val(rr, 2)); grid.append([val(rr, 3 + j) for j in range(len(cols))]); rr += 1
                out[section] = dict(row_values=rowv, col_values=cols, grid=np.array(grid, dtype=float))
                r = rr; continue
            if section == "E":
                out["headroom"] = dict(price_to_zero_ebitda=val(r + 1, 3), cost_to_zero_ebitda=val(r + 2, 3))
                break
            hdr = r + 2                       # 'Current model' row, then header row, then selector row
            names = [val(hdr, c) for c in range(3, 13) if val(hdr, c)]
            rr, data = hdr + 2, []
            while isinstance(val(rr, 2), (int, float)):
                data.append(dict(value=val(rr, 2), **{n: val(rr, 3 + j) for j, n in enumerate(names)})); rr += 1
            out[section] = data
            r = rr; continue
        r += 1
    return out


### Cell 4 – The new screen (`stress_view.py`)

In [ ]:
%%writefile stress_view.py
"""
stress_view.py  -  STEP 8b: the 'Stress & Monte Carlo' screen.

Figure builders are plain functions (testable); render_stress() draws the page in Streamlit.
"""
from __future__ import annotations

import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from engine.inputs import from_dict
from engine.stress import (DEFAULT_STEPS, LEVERS, McRanges, headroom, monte_carlo, one_way, outcome, scenario_table,
                           tornado, two_way)

BLUE, GREEN, RED, ORANGE, NAVY = "#5B9BD5", "#70AD47", "#C00000", "#ED7D31", "#1F3864"
FMT = {"price": "{:+.0%}", "cost": "{:+.0%}", "speed": "{:.2f}x", "delay": "+{:d} Q"}


def _lay(fig, title, h=340):
    fig.update_layout(title=dict(text=title, x=0.01, font=dict(size=14)), height=h, margin=dict(l=10, r=10, t=45, b=10),
                      plot_bgcolor="white", legend=dict(orientation="h", y=-0.2))
    fig.update_yaxes(gridcolor="#EDEDED")
    return fig


def compute_tables(project) -> dict:
    """Everything that does not depend on the sliders (~150 model runs)."""
    return dict(
        scenarios=scenario_table(project),
        one_way={lv: one_way(project, lv) for lv in DEFAULT_STEPS},
        tornado=tornado(project),
        irr_grid=two_way(project, "cost", [0.0, 0.05, 0.10, 0.15, 0.20], "price", [-0.10, -0.05, 0.0, 0.05, 0.10], "developer_irr"),
        pat_grid=two_way(project, "cost", [0.0, 0.05, 0.10, 0.15, 0.20], "price", [-0.10, -0.05, 0.0, 0.05, 0.10], "pat"),
        peak_grid=two_way(project, "speed", [0.75, 1.0, 1.25, 1.5, 2.0], "delay", [0, 1, 2, 4], "peak"),
        headroom=headroom(project))


def fig_one_way(rows, lever, hurdle):
    x = [FMT[lever].format(int(r["value"]) if lever == "delay" else r["value"]) for r in rows]
    y = [r["developer_irr"] for r in rows]
    f = go.Figure(go.Scatter(x=x, y=y, mode="lines+markers+text", text=[f"{v:.0%}" if v is not None else "n/a" for v in y],
                             textposition="top center", line=dict(color=NAVY, width=3)))
    f.add_hline(y=hurdle, line_dash="dash", line_color=RED, annotation_text=f"hurdle {hurdle:.0%}")
    f.update_yaxes(tickformat=".0%")
    return _lay(f, LEVERS[lever], 280)


def fig_tornado(rows):
    labels = [r["lever"] + f"  ({FMT_KEY(r)})" for r in rows]
    f = go.Figure()
    f.add_bar(y=labels, x=[(r["down"] or 0) - r["base"] for r in rows], orientation="h", name="Downside", marker_color=RED,
              text=[f"{r['down']:.0%}" if r["down"] is not None else "n/a" for r in rows], textposition="outside")
    f.add_bar(y=labels, x=[(r["up"] or 0) - r["base"] for r in rows], orientation="h", name="Upside", marker_color=GREEN,
              text=[f"{r['up']:.0%}" if r["up"] is not None else "n/a" for r in rows], textposition="outside")
    f.update_layout(barmode="overlay"); f.update_xaxes(tickformat="+.0%", title="change in developer IRR vs base")
    return _lay(f, f"What moves developer IRR the most? (base {rows[0]['base']:.0%})", 320)


def FMT_KEY(r):
    lever = next(k for k, v in LEVERS.items() if v == r["lever"])
    show = lambda v: FMT[lever].format(int(v) if lever == "delay" else v)
    return f"{show(r['down_value'])} / {show(r['up_value'])}"


def fig_heat(grid, rows, cols, row_fmt, col_fmt, title, value_fmt, good_high=True, y_title="", x_title=""):
    f = px.imshow(grid, x=[col_fmt.format(c) for c in cols], y=[row_fmt.format(r) for r in rows], aspect="auto",
                  color_continuous_scale="RdYlGn" if good_high else "RdYlGn_r", text_auto=False)
    f.update_traces(text=[[value_fmt.format(v) for v in row] for row in grid], texttemplate="%{text}",
                    hovertemplate=f"{y_title} %{{y}} · {x_title} %{{x}}: %{{text}}<extra></extra>")
    f.update_layout(coloraxis_showscale=False); f.update_xaxes(title=x_title, side="top"); f.update_yaxes(title=y_title)
    return _lay(f, title, 320)


def fig_mc_hist(mc, threshold):
    irr = mc["irr"][~np.isnan(mc["irr"])]
    f = go.Figure(go.Histogram(x=irr, nbinsx=40, marker_color=BLUE, hovertemplate="IRR %{x:.0%}: %{y} runs<extra></extra>"))
    for v, lab, col in [(threshold, f"threshold {threshold:.0%}", RED), (mc["irr_p10"], "P10", ORANGE), (mc["irr_p50"], "P50", NAVY),
                        (mc["irr_p90"], "P90", GREEN)]:
        f.add_vline(x=v, line_dash="dash", line_color=col, annotation_text=lab, annotation_position="top")
    f.update_xaxes(tickformat=".0%", title="developer IRR"); f.update_yaxes(title="number of runs")
    return _lay(f, f"Developer IRR across {mc['n']:,} random scenarios", 360)


def fig_mc_exceedance(mc, threshold):
    irr = np.sort(mc["irr"][~np.isnan(mc["irr"])])
    prob = 1 - np.arange(len(irr)) / len(irr)
    f = go.Figure(go.Scatter(x=irr, y=prob, mode="lines", line=dict(color=NAVY, width=3),
                             hovertemplate="P(IRR ≥ %{x:.0%}) = %{y:.0%}<extra></extra>"))
    p_thr = float(np.mean(irr >= threshold)) if len(irr) else 0.0
    f.add_vline(x=threshold, line_dash="dash", line_color=RED)
    f.add_annotation(x=threshold, y=p_thr, text=f"{p_thr:.0%} chance ≥ {threshold:.0%}", showarrow=True, arrowhead=2)
    f.update_xaxes(tickformat=".0%", title="developer IRR at least …"); f.update_yaxes(tickformat=".0%", title="probability")
    return _lay(f, "Probability that developer IRR is at least X", 360)


def fig_mc_scatter(mc):
    f = go.Figure(go.Scatter(x=mc["peak"], y=mc["irr"], mode="markers", marker=dict(size=5, color=mc["samples"]["price"],
                  colorscale="RdYlGn", colorbar=dict(title="price<br>shock", tickformat=".0%"), opacity=0.7),
                  hovertemplate="peak ₹%{x:,.1f} Cr · IRR %{y:.0%}<extra></extra>"))
    f.update_xaxes(title="peak investment (₹ Cr)"); f.update_yaxes(tickformat=".0%", title="developer IRR")
    return _lay(f, "Return vs funding need for every run (colour = price shock)", 360)


def render_stress(P, fingerprint: str):
    import streamlit as st

    @st.cache_data(show_spinner="Running the stress tables (about 150 model runs)…", max_entries=8)
    def cached(fp: str):
        return compute_tables(from_dict(json.loads(fp)))

    st.header("Stress testing & Monte Carlo")
    st.caption("Levers act on top of the active scenario (page 1), exactly like the Excel STRESS sheet. Your inputs are never changed.")
    hurdle = P.finance.benchmarks["developer_irr"]

    # ---------- A. live sliders
    with st.container(border=True):
        st.subheader("A · Live what-if")
        c = st.columns(4)
        price = c[0].slider("Sales price shock", -30, 30, 0, 1, format="%d%%") / 100
        cost = c[1].slider("Construction cost shock", -10, 40, 0, 1, format="%d%%") / 100
        speed = c[2].slider("Sales period multiplier", 0.50, 2.50, 1.00, 0.05, format="%.2fx")
        delay = c[3].slider("Extra delay (quarters)", 0, 8, 0, 1)
        base, s = outcome(P), outcome(P, price=price, cost=cost, speed=speed, delay=delay)
        if not s["ok"]:
            st.warning("This combination breaks a model check (e.g. the project runs past 40 quarters): " + "; ".join(s["problems"][:2]))
        m = st.columns(5)
        pct = lambda v: "n/a" if v is None else f"{v:.1%}"
        dpp = lambda a, b: None if (a is None or b is None) else f"{(a - b) * 100:+.1f} pts"
        m[0].metric("Developer IRR", pct(s["developer_irr"]), dpp(s["developer_irr"], base["developer_irr"]))
        m[1].metric("Developer NPV", f"₹{s['developer_npv']:,.1f} Cr", f"{s['developer_npv'] - base['developer_npv']:+,.1f}")
        m[2].metric("PAT", f"₹{s['pat']:,.1f} Cr", f"{s['pat'] - base['pat']:+,.1f}")
        m[3].metric("Peak investment", f"₹{s['peak']:,.1f} Cr", f"{s['peak'] - base['peak']:+,.1f}", delta_color="inverse")
        m[4].metric("Payback", f"Q{s['payback_q']}" if s["payback_q"] else "never",
                    f"{(s['payback_q'] or 99) - (base['payback_q'] or 99):+d} qtrs" if s["payback_q"] and base["payback_q"] else None, delta_color="inverse")
        verdict = s["developer_irr"] is not None and s["developer_irr"] >= hurdle
        (st.success if verdict else st.error)(f"Developer IRR {'stays above' if verdict else 'falls below'} the {hurdle:.0%} hurdle in this case.")
        k = 32
        f = go.Figure()
        f.add_scatter(y=np.cumsum(base["developer_cf"])[:k], x=[f"Q{i}" for i in range(1, k + 1)], name="Base case", line=dict(color="#A5A5A5", width=3))
        f.add_scatter(y=np.cumsum(s["developer_cf"])[:k], x=[f"Q{i}" for i in range(1, k + 1)], name="Stressed", line=dict(color=RED, width=3))
        st.plotly_chart(_lay(f, "Developer cumulative cash (funding curve): base vs stressed (₹ Cr)", 320), width="stretch")

    T = cached(fingerprint)

    # ---------- B. scenarios + tornado
    c1, c2 = st.columns(2)
    with c1:
        st.subheader("B · Scenario comparison")
        df = pd.DataFrame(T["scenarios"])[["scenario", "gdv", "ebitda", "pat", "developer_irr", "developer_npv", "peak", "payback_q"]]
        df.columns = ["Scenario", "GDV ₹Cr", "EBITDA ₹Cr", "PAT ₹Cr", "Developer IRR", "Dev NPV ₹Cr", "Peak ₹Cr", "Payback"]
        df["Developer IRR"] = df["Developer IRR"].map(lambda v: "n/a" if v is None else f"{v:.1%}")
        df["Payback"] = df["Payback"].map(lambda q: f"Q{q}" if q else "never")
        st.dataframe(df.round(1), hide_index=True, width="stretch")
    with c2:
        st.plotly_chart(fig_tornado(T["tornado"]), width="stretch")

    # ---------- C. one-way
    st.subheader("C · One lever at a time – developer IRR")
    cols = st.columns(4)
    for col, lv in zip(cols, DEFAULT_STEPS):
        col.plotly_chart(fig_one_way(T["one_way"][lv], lv, hurdle), width="stretch")
    with st.expander("Show the one-way tables (IRR, NPV, PAT, EBITDA, peak, payback)"):
        for lv in DEFAULT_STEPS:
            d = pd.DataFrame(T["one_way"][lv])[["value", "developer_irr", "developer_npv", "pat", "ebitda", "peak", "payback_q"]]
            d.columns = [LEVERS[lv], "Developer IRR", "Dev NPV ₹Cr", "PAT ₹Cr", "EBITDA ₹Cr", "Peak ₹Cr", "Payback Q"]
            st.dataframe(d.round(4), hide_index=True, width="stretch")

    # ---------- D. heat maps
    st.subheader("D · Two levers together – heat maps")
    h = st.columns(3)
    pr, co_, sp, dl = [-0.10, -0.05, 0.0, 0.05, 0.10], [0.0, 0.05, 0.10, 0.15, 0.20], [0.75, 1.0, 1.25, 1.5, 2.0], [0, 1, 2, 4]
    h[0].plotly_chart(fig_heat(T["irr_grid"], co_, pr, "{:+.0%}", "{:+.0%}", "Developer IRR: price × cost", "{:.0%}",
                               y_title="cost", x_title="price"), width="stretch")
    h[1].plotly_chart(fig_heat(T["pat_grid"], co_, pr, "{:+.0%}", "{:+.0%}", "PAT (₹ Cr): price × cost", "{:,.0f}",
                               y_title="cost", x_title="price"), width="stretch")
    h[2].plotly_chart(fig_heat(T["peak_grid"], sp, dl, "{:.2f}x", "+{} Q", "Peak investment (₹ Cr): speed × delay", "{:,.1f}",
                               good_high=False, y_title="speed", x_title="delay"), width="stretch")

    # ---------- E. headroom
    st.subheader("E · Headroom – how much can go wrong?")
    hr = T["headroom"]
    pct0 = lambda v: "beyond the tested range" if v is None else f"{v:+.1%}"
    e = st.columns(4)
    e[0].metric("Price shock → EBITDA = 0", pct0(hr["price_to_zero_ebitda"]))
    e[1].metric("Cost overrun → EBITDA = 0", pct0(hr["cost_to_zero_ebitda"]))
    e[2].metric(f"Price shock → IRR = {hurdle:.0%}", pct0(hr["price_to_hurdle"]))
    e[3].metric(f"Cost overrun → IRR = {hurdle:.0%}", pct0(hr["cost_to_hurdle"]))
    st.caption(("Sales speed: IRR stays above the hurdle up to 3.0x slower. " if hr["speed_to_hurdle"] is None
                else f"Sales speed: IRR reaches the hurdle at {hr['speed_to_hurdle']:.2f}x. ") +
               ("Delay: IRR stays above the hurdle for every delay the model can test (up to 12 quarters)." if hr["delay_to_hurdle"] is None
                else f"Delay: IRR falls below the hurdle at +{hr['delay_to_hurdle']} quarters."))

    # ---------- F. Monte Carlo
    st.subheader("F · Monte Carlo – probabilities instead of single answers")
    st.caption("Each run picks a random price shock, cost shock, sales speed and delay from the ranges below "
               "(triangular: lowest · most likely · highest), runs the full model, and records the results.")
    with st.form("mc_form"):
        r = McRanges()
        c = st.columns(3)
        def tri(col, label, t, scale, fmt):
            lo = col.number_input(f"{label} – lowest", value=t[0] * scale, format=fmt)
            ml = col.number_input(f"{label} – most likely", value=t[1] * scale, format=fmt)
            hi = col.number_input(f"{label} – highest", value=t[2] * scale, format=fmt)
            return (lo / scale, ml / scale, hi / scale)
        price_r = tri(c[0], "Price shock %", r.price, 100, "%.1f")
        cost_r = tri(c[1], "Cost shock %", r.cost, 100, "%.1f")
        speed_r = tri(c[2], "Sales period ×", r.speed, 1, "%.2f")
        st.write("Probability of an extra delay")
        d = st.columns(4)
        probs = {q: d[i].number_input(f"+{q} quarter(s) %", value=p * 100, min_value=0.0, max_value=100.0, step=5.0) / 100
                 for i, (q, p) in enumerate(r.delay_probs.items())}
        c = st.columns(3)
        n = c[0].selectbox("Number of runs", [250, 500, 1000, 2000], index=2)
        seed = int(c[1].number_input("Random seed (same seed = same answer)", value=42, step=1))
        go_ = c[2].form_submit_button("▶ Run Monte Carlo", type="primary")
    bad = [lab for lab, t in [("price", price_r), ("cost", cost_r), ("speed", speed_r)] if not (t[0] <= t[1] <= t[2])]
    if bad:
        st.error(f"For {', '.join(bad)}: lowest ≤ most likely ≤ highest is required.")
    elif sum(probs.values()) <= 0:
        st.error("Delay probabilities must add up to more than 0%.")
    elif go_:
        bar = st.progress(0.0, text="Running…")
        res = monte_carlo(P, McRanges(price_r, cost_r, speed_r, probs), n=n, seed=seed, progress=lambda x: bar.progress(x, text=f"Running… {x:.0%}"))
        bar.empty()
        st.session_state.mc = dict(fp=fingerprint, res=res)
    mc_state = st.session_state.get("mc")
    if mc_state:
        mc = mc_state["res"]
        if mc_state["fp"] != fingerprint:
            st.warning("These results were calculated before your latest input changes – press ▶ Run Monte Carlo again.")
        thr = st.slider("Show the probability of developer IRR being at least…", 0, 150, int(round(mc["hurdle"] * 100)), 1, format="%d%%") / 100
        irr_ok = mc["irr"][~np.isnan(mc["irr"])]
        p_thr = float(np.mean(irr_ok >= thr)) if len(irr_ok) else 0.0
        k = st.columns(4)
        k[0].metric(f"P(developer IRR ≥ {thr:.0%})", f"{p_thr:.1%}")
        k[1].metric("P(NPV > 0)", f"{mc['p_npv_positive']:.1%}")
        k[2].metric("P(loss: PAT < 0)", f"{mc['p_loss']:.1%}")
        k[3].metric("Peak investment P90", f"₹{mc['peak_p90']:,.1f} Cr", f"worst ₹{mc['peak_max']:,.1f} Cr", delta_color="off")
        st.info(f"In {mc['n']:,} random scenarios the developer IRR is between **{mc['irr_p10']:.0%}** (P10 – only 1 in 10 runs is worse) "
                f"and **{mc['irr_p90']:.0%}** (P90), with a median of **{mc['irr_p50']:.0%}**. "
                f"There is a **{p_thr:.0%}** chance it is at least {thr:.0%}." +
                (f" ({mc['failed_runs']} runs broke a model check.)" if mc["failed_runs"] else ""), icon="🎲")
        c1, c2 = st.columns(2)
        c1.plotly_chart(fig_mc_hist(mc, thr), width="stretch")
        c2.plotly_chart(fig_mc_exceedance(mc, thr), width="stretch")
        st.plotly_chart(fig_mc_scatter(mc), width="stretch")
        st.dataframe(pd.DataFrame({"": ["P10 (pessimistic)", "P50 (median)", "P90 (optimistic)"],
                                   "Developer IRR": [f"{mc['irr_p10']:.1%}", f"{mc['irr_p50']:.1%}", f"{mc['irr_p90']:.1%}"],
                                   "Developer NPV ₹Cr": [round(mc["npv_p10"], 1), round(mc["npv_p50"], 1), round(mc["npv_p90"], 1)],
                                   "PAT ₹Cr": [round(mc["pat_p10"], 1), round(mc["pat_p50"], 1), round(mc["pat_p90"], 1)]}),
                     hide_index=True)


### Cell 5 – Updated app (`app.py`) – adds **S · Stress & Monte Carlo** after the dashboard

In [ ]:
%%writefile app.py
"""
app.py  -  STEP 6: the input screens of the Real Estate Feasibility Engine (Streamlit).

Run with:   streamlit run app.py
Every screen edits ONE Project object kept in memory (st.session_state). After every change the whole
engine re-runs (a fraction of a second), and any problem is shown right next to the field / row that caused it.
"""
from __future__ import annotations

import copy
import json
from datetime import date

import pandas as pd
import streamlit as st

from engine.inputs import (
    BAR_AUTHORITIES, BAR_LAND_USES, BASIC_CURVES, CATEGORIES, COMPOUNDING, CONS_MODES, CONSTRUCTION_HEADS,
    CONTINGENCY_BASES, COST_CURVES, CRORE, DEAL_TYPES, EDC_BASES, ENTITIES, HORIZON_Q, LAND_CURVES, LAUNCH_MODES,
    LINK_EVENTS, PROJECT_TYPES, SPARE_BASES, SQFT_PER_SQYD, SQYD_PER_ACRE, SQYD_PER_BIGHA, TRIGGERS, ZONES,
    CostHead, ClpPlan, Milestone, Product, Scenario, Timing, check_inputs, from_dict, sample_project, to_dict,
)
from engine.cashflow import run_model
from dashboard import render_dashboard
from stress_view import render_stress

st.set_page_config(page_title="Real Estate Feasibility Engine", page_icon="🏗️", layout="wide")

# ============================================================================ state & engine run
if "project" not in st.session_state:
    st.session_state.project = sample_project()
P = st.session_state.project


def rerun_after_edit(editor_key: str):
    """Commit pattern for editable tables: the project is already updated -> clear the table's edit
    memory and redraw everything from the project."""
    st.session_state.pop(editor_key, None)
    st.rerun()


FINGERPRINT = json.dumps(to_dict(P), sort_keys=True, default=str)   # project state used for this run's results
issues = check_inputs(P)
try:
    RES = run_model(P)
except Exception as exc:                      # an unforeseen input combination – never crash the screens
    RES, ENGINE_ERROR = None, str(exc)
else:
    ENGINE_ERROR = ""

PAGE_SECTIONS = {
    "0 · Dashboard": [],
    "S · Stress & Monte Carlo": [],
    "1 · Project & land": ["Project", "Land"],
    "2 · Scenarios & zones": ["Scenarios", "Zones"],
    "3 · Products": ["Products"],
    "4 · Product timing": ["Timing"],
    "5 · Cost heads": ["Costs"],
    "6 · Calculators": ["BAR", "AIF", "Initial investment"],
    "7 · Payment plans": ["CLP"],
    "8 · Tax & benchmarks": ["Finance"],
    "9 · Checks & summary": [],
}


def issues_for(section: str, item: str | None = None) -> list:
    return [i for i in issues if i.section == section and (item is None or i.item == item)]


def show(section: str, item: str):
    """Show the error(s) for one field directly under it."""
    for i in issues_for(section, item):
        st.error(i.message, icon="⚠️")


def row_status(section: str, item: str) -> str:
    msgs = [i.message.split(": ", 1)[-1] for i in issues_for(section, item)]
    return "✔" if not msgs else "⚠ " + " | ".join(msgs)


def pct_in(label, value, key=None, help=None, max_value=100.0, step=0.25):
    """Percentage field: shown as 32 (%) but stored as 0.32."""
    return st.number_input(label, value=round(float(value) * 100, 6), min_value=-100.0, max_value=max_value,
                           step=step, format="%.2f", key=key, help=help) / 100


# ============================================================================ sidebar
with st.sidebar:
    st.title("🏗️ Feasibility Engine")
    st.caption(f"{P.info.name} · {P.info.project_type} · {P.info.deal_type} · {P.info.scenario}")
    labels = []
    for page, secs in PAGE_SECTIONS.items():
        n = sum(1 for i in issues if i.section in secs)
        labels.append(f"{page}  ⚠️ {n}" if n else page)
    choice = st.radio("Go to", labels, label_visibility="collapsed")
    page = choice.split("  ⚠️")[0]

    st.divider()
    engine_problems = RES["problems"] if RES else []
    if not issues and RES and not engine_problems:
        st.success("✔ ALL CHECKS PASSED")
    else:
        n_problems = len(issues) + sum(1 for p in engine_problems if not any(p.endswith(i.message) for i in issues))
        st.error(f"✖ {n_problems or 1} issue(s) – see page 9")
    if RES:
        m, pnl = RES["cashflow"].metrics, RES["cashflow"].pnl
        c1, c2 = st.columns(2)
        c1.metric("GDV", f"₹{pnl['revenue']:,.0f} Cr")
        c2.metric("EBITDA", f"{pnl['ebitda_margin']:.1%}")
        c1.metric("Developer IRR", "n/a" if m["developer_irr"] is None else f"{m['developer_irr']:.1%}")
        c2.metric("Peak funding", f"₹{m['peak_investment']:,.1f} Cr")
        st.caption(f"Payback: {'Q' + str(m['payback_q']) if m['payback_q'] else 'never'} · "
                   f"PAT ₹{pnl['pat']:,.1f} Cr · {RES['timeline'].last_q} quarters")

    st.divider()
    st.download_button("💾 Save project file", json.dumps(to_dict(P), indent=2, ensure_ascii=False),
                       file_name=f"{P.info.name.split(' (')[0].replace(' ', '_')}.json", mime="application/json",
                       width="stretch")
    up = st.file_uploader("📂 Open project file", type="json")
    if up is not None and st.session_state.get("loaded_file") != up.file_id:
        try:
            st.session_state.project = from_dict(json.load(up))
            st.session_state.loaded_file = up.file_id
            st.rerun()
        except Exception as exc:
            st.error(f"Could not open this file: {exc}")
    if st.button("↺ Reset to sample project", width="stretch"):
        st.session_state.project = sample_project()
        st.rerun()

# ============================================================================ page 0: dashboard
if page.startswith("0"):
    if RES:
        render_dashboard(P, RES)
    else:
        st.error("The dashboard cannot be drawn until the errors on the other pages are fixed (see '9 · Checks & summary').")

# ============================================================================ stress testing & Monte Carlo
elif page.startswith("S"):
    if RES and not RES["problems"]:
        render_stress(P, FINGERPRINT)
    else:
        st.error("Stress tests need a model without errors – fix the issues listed on '9 · Checks & summary' first.")

# ============================================================================ page 1: project & land
elif page.startswith("1"):
    st.header("1 · Project & land")
    st.caption("Fields with ⚠️ underneath need fixing. All amounts in ₹; land in sq yd.")
    a, b = st.columns(2)
    with a:
        st.subheader("Project")
        P.info.name = st.text_input("Project name", P.info.name)
        P.info.location = st.text_input("Location", P.info.location)
        P.info.project_type = st.selectbox("Project type", PROJECT_TYPES, PROJECT_TYPES.index(P.info.project_type),
                                           help="Only products whose zone belongs to this type are calculated (Mixed = all).")
        P.info.start_date = st.date_input("Project start = Q1 (quarter-end date)", P.info.start_date,
                                          min_value=date(2020, 1, 1), max_value=date(2045, 12, 31), format="DD/MM/YYYY")
        show("Project", "start_date")
        P.info.deal_type = st.radio("Deal type", DEAL_TYPES, DEAL_TYPES.index(P.info.deal_type), horizontal=True)
        P.info.entity = st.radio("Taxable entity", ENTITIES, ENTITIES.index(P.info.entity), horizontal=True)
        names = [s.name for s in P.scenarios]
        P.info.scenario = st.selectbox("Active scenario", names, names.index(P.info.scenario) if P.info.scenario in names else 0)
        show("Project", "scenario")
        ov = st.number_input("Lifecycle for ROCE / CAGR (years) – 0 = use project span", value=float(P.info.lifecycle_override_years or 0),
                             min_value=0.0, step=0.5)
        P.info.lifecycle_override_years = ov or None
    with b:
        st.subheader("Land")
        L = P.land
        L.area_sqyd = st.number_input("Land area (sq yd)", value=float(L.area_sqyd), min_value=0.0, step=100.0)
        st.caption(f"= {L.area_sqyd * SQFT_PER_SQYD:,.0f} sq ft · {L.area_sqyd / SQYD_PER_ACRE:,.2f} acres · "
                   f"{L.area_sqyd / SQYD_PER_BIGHA:,.2f} bigha")
        L.market_rate = st.number_input("Current market rate (₹ / sq yd)", value=float(L.market_rate), min_value=0.0, step=500.0)
        L.dlc_rate_sqm = st.number_input("DLC / circle rate (₹ / sq m)", value=float(L.dlc_rate_sqm), min_value=0.0, step=100.0)
        show("Land", "land_rates")
        if P.info.deal_type == "JV":
            L.lo_share = pct_in("Landowner revenue share (%)", L.lo_share)
            show("Land", "lo_share")
            L.stamp_duty_jv = pct_in("Stamp duty + registration on JV (% of DLC value)", L.stamp_duty_jv, step=0.05)
            if RES:
                cr = RES["costs"]
                lo_px = cr.land_cost * CRORE / L.area_sqyd if L.area_sqyd else 0
                st.info(f"Landowner effectively gets ₹{lo_px:,.0f} / sq yd = "
                        f"{lo_px / L.market_rate - 1:+.1%} vs today's market rate.", icon="💡")
        else:
            L.purchase_rate = st.number_input("Purchase rate (₹ / sq yd)", value=float(L.purchase_rate), min_value=0.0, step=500.0)
            show("Land", "purchase_rate")
            L.stamp_duty_outright = pct_in("Stamp duty + registration – outright (%)", L.stamp_duty_outright, step=0.05)
            c1, c2, c3 = st.columns(3)
            L.payment_curve = c1.selectbox("Land payment curve", LAND_CURVES, LAND_CURVES.index(L.payment_curve))
            L.payment_start_q = int(c2.number_input("Start quarter", value=int(L.payment_start_q), min_value=1, max_value=HORIZON_Q))
            L.payment_end_q = int(c3.number_input("End quarter", value=int(L.payment_end_q), min_value=1, max_value=HORIZON_Q,
                                                  disabled=L.payment_curve == "Upfront"))
            show("Land", "payment")

# ============================================================================ page 2: scenarios & zones
elif page.startswith("2"):
    st.header("2 · Scenarios & development zones")
    st.subheader("Scenarios")
    st.caption("Sales period multiplier 1.5 = products take 50% longer to sell. Delay pushes every fixed launch / construction start later.")
    df = pd.DataFrame([{"Scenario": s.name, "Sales period ×": s.sales_period_mult, "Price change %": s.price_change * 100,
                        "Construction cost change %": s.cost_change * 100, "Delay (quarters)": s.delay_q,
                        "Check": row_status("Scenarios", s.name)} for s in P.scenarios])
    ed = st.data_editor(df, key="ed_scen", hide_index=True, width="stretch", num_rows="dynamic",
                        column_config={"Sales period ×": st.column_config.NumberColumn(min_value=0.1, step=0.05, format="%.2f"),
                                       "Delay (quarters)": st.column_config.NumberColumn(min_value=0, step=1),
                                       "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        P.scenarios = [Scenario(str(r["Scenario"] or f"Scenario {i+1}"), float(r["Sales period ×"] or 1), float(r["Price change %"] or 0) / 100,
                                float(r["Construction cost change %"] or 0) / 100, int(r["Delay (quarters)"] or 0))
                       for i, r in ed.iterrows()]
        rerun_after_edit("ed_scen")
    show("Scenarios", "names")

    st.subheader("Development zones")
    used = {"Villa": ["Villa Development"], "Apartments": ["Apartment Development"], "Plotted": ["Plotted Development"]}.get(P.info.project_type, ZONES)
    df = pd.DataFrame([{"Zone": z.name, "Share of land %": z.share * 100,
                        "Land area (sq ft)": round(z.share * P.land.area_sqyd * SQFT_PER_SQYD),
                        "In this project type?": "Yes" if (z.name in used or z.name == "Other / Clubhouse") else "No"} for z in P.zones])
    ed = st.data_editor(df, key="ed_zone", hide_index=True, width="stretch", num_rows="fixed",
                        column_config={"Zone": st.column_config.TextColumn(disabled=True),
                                       "Land area (sq ft)": st.column_config.NumberColumn(disabled=True, format="%d"),
                                       "In this project type?": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for z, (_, r) in zip(P.zones, ed.iterrows()):
            z.share = float(r["Share of land %"] or 0) / 100
        rerun_after_edit("ed_zone")
    st.caption(f"Total share: {sum(z.share for z in P.zones):.1%}")
    show("Zones", "zones")

# ============================================================================ page 3: products
elif page.startswith("3"):
    st.header("3 · Product mix")
    st.caption("One row per product. Add a row with the ➕ at the bottom of the table; tick **Include** to use it. "
               "**Active** is automatic: included AND its zone fits the project type.")
    active = {p.name for p in P.active_products()}
    plans = [pl.name for pl in P.plans]
    rev = RES["curves"].revenue if RES else {}
    df = pd.DataFrame([{
        "Include": p.include, "Product": p.name, "Zone": p.zone, "Category": p.category,
        "Area (sq ft)": p.area_sqft, "Sales rate (₹/sq ft)": p.rate, "Construction (₹/sq ft)": p.construction_rate,
        "Launch Q (fixed)": p.launch_q, "Sales period (qtrs)": p.sales_period_q, "Sales curve": p.sales_curve,
        "Payment plan": p.clp_plan, "Construction start Q (fixed)": p.construction_start_q,
        "Construction (qtrs)": p.construction_duration_q, "Construction curve": p.construction_curve,
        "Active": "Yes" if p.name in active else ("Excluded by type" if p.include else "Off"),
        "Revenue (₹ Cr)": round(rev.get(p.name, 0.0), 2), "Check": row_status("Products", p.name)} for p in P.products])
    num = lambda **k: st.column_config.NumberColumn(**k)
    ed = st.data_editor(df, key="ed_prod", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Include": st.column_config.CheckboxColumn(default=True),
        "Zone": st.column_config.SelectboxColumn(options=ZONES, required=True, default="Apartment Development"),
        "Category": st.column_config.SelectboxColumn(options=CATEGORIES, required=True, default="Apartment"),
        "Area (sq ft)": num(min_value=0, step=1000, format="%d"), "Sales rate (₹/sq ft)": num(min_value=0, step=100, format="%d"),
        "Construction (₹/sq ft)": num(min_value=0, step=50, format="%d"),
        "Launch Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1), "Sales period (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Sales curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="Even"),
        "Payment plan": st.column_config.SelectboxColumn(options=plans, required=True, default=plans[0] if plans else None),
        "Construction start Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="S-curve"),
        "Active": st.column_config.TextColumn(disabled=True), "Revenue (₹ Cr)": num(disabled=True, format="%.2f"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        old = {i: p for i, p in enumerate(P.products)}
        new = []
        for i, r in ed.reset_index(drop=True).iterrows():
            v = lambda col, d: d if pd.isna(r[col]) else r[col]
            prev = old.get(i)
            name = str(v("Product", f"New product {i+1}"))
            prod = Product(name, v("Zone", "Apartment Development"), v("Category", "Apartment"), bool(v("Include", True)),
                           float(v("Area (sq ft)", 0)), float(v("Sales rate (₹/sq ft)", 0)), float(v("Construction (₹/sq ft)", 0)),
                           int(v("Launch Q (fixed)", 1)), int(v("Sales period (qtrs)", 1)), v("Sales curve", "Even"),
                           v("Payment plan", plans[0] if plans else ""), int(v("Construction start Q (fixed)", 1)),
                           int(v("Construction (qtrs)", 1)), v("Construction curve", "S-curve"),
                           copy.deepcopy(prev.timing) if prev else Timing())
            if prev and prev.name != name:                       # renamed -> keep links pointing to it
                for q in P.products:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
                for q in new:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
            new.append(prod)
        P.products = new
        rerun_after_edit("ed_prod")
    show("Products", "names"); show("Products", "active")
    if RES:
        st.caption(f"Active saleable area {RES['costs'].total_sa:,.0f} sq ft · GDV ₹{RES['costs'].total_revenue:,.2f} Cr · "
                   f"construction ₹{RES['costs'].total_construction:,.2f} Cr")

# ============================================================================ page 4: product timing
elif page.startswith("4"):
    st.header("4 · Product timing – how each launch and construction start is decided")
    st.caption("Fixed = uses the quarter typed on page 3. Linked = X quarters after an event of a product listed ABOVE it "
               "(put the lead product first). Offset 0 = same quarter, 1 = next quarter, −1 = one before.")
    names = [p.name for p in P.products]
    notes = {n: t.note for n, t in RES["timeline"].products.items()} if RES else {}
    df = pd.DataFrame([{
        "Product": p.name, "Launch timing": p.timing.launch_mode, "Launch – linked product": p.timing.launch_link_product,
        "Launch – event": p.timing.launch_link_event, "Launch offset": p.timing.launch_offset,
        "Construction timing": p.timing.cons_mode, "Construction – linked product": p.timing.cons_link_product,
        "Construction – event": p.timing.cons_link_event, "Construction offset": p.timing.cons_offset,
        "Result": notes.get(p.name, "— (not active)"), "Check": row_status("Timing", p.name)} for p in P.products])
    sel = lambda opts: st.column_config.SelectboxColumn(options=opts)
    ed = st.data_editor(df, key="ed_time", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Product": st.column_config.TextColumn(disabled=True), "Launch timing": sel(LAUNCH_MODES),
        "Launch – linked product": sel(names), "Launch – event": sel(LINK_EVENTS),
        "Launch offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Construction timing": sel(CONS_MODES), "Construction – linked product": sel(names), "Construction – event": sel(LINK_EVENTS),
        "Construction offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Result": st.column_config.TextColumn(disabled=True, width="large"), "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for p, (_, r) in zip(P.products, ed.iterrows()):
            n = lambda col, d=None: d if pd.isna(r[col]) else r[col]
            p.timing = Timing(n("Launch timing", "Fixed quarter"), n("Launch – linked product"), n("Launch – event"), int(n("Launch offset", 0)),
                              n("Construction timing", "Fixed quarter"), n("Construction – linked product"), n("Construction – event"),
                              int(n("Construction offset", 0)))
        rerun_after_edit("ed_time")
    if RES:
        for e in RES["timeline"].errors: st.error(e, icon="⚠️")
        tl = RES["timeline"]
        st.subheader("Timeline (S = selling · C = construction · # = both)")
        rows = []
        for t in tl.products.values():
            rows.append({"Product": t.name, **{f"Q{q}": ("#" if (t.launch <= q <= t.sales_end and t.cons_start <= q <= t.cons_end)
                                                      else "S" if t.launch <= q <= t.sales_end else "C" if t.cons_start <= q <= t.cons_end else "")
                                               for q in range(1, max(tl.last_q, 1) + 1)}})
        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")

# ============================================================================ page 5: cost heads
elif page.startswith("5"):
    st.header("5 · Cost heads")
    st.caption("Change rates, switch heads on/off and pick how each cost is spread over time. Grey 'Auto' rows are calculated "
               "elsewhere (land page, product table, calculators). Percent inputs are typed as % (4 = 4%).")
    META = {  # key: (how it is calculated, unit of input 1, unit of input 2)
        "land": ("JV: landowner share × revenue | Outright: price × area", "Auto", ""), "sd": ("From land page", "Auto", ""),
        "brok": ("% × land value", "%", ""), "ovh": ("% × total revenue", "%", ""),
        "mkt": ("Input 1 % × built-up revenue + Input 2 % × plotted revenue", "%", "%"),
        "legal": ("₹/sq ft × total saleable area", "₹/sq ft", ""), "ews": ("Loss ₹/sq ft × EWS share × saleable area", "₹/sq ft", "%"),
        "cont": ("% × base chosen in 'Basis'", "%", ""), "edc": ("Input 1 per 'Basis'", "per basis", ""),
        "hotel": ("Fee per key (₹) × number of keys", "₹/key", "keys"), "furn": ("₹/sq ft × studio area", "₹/sq ft", ""),
        "gst": ("(Commercial + plot revenue) × unsold % × GST % × developer share", "% GST", "% unsold"),
        "aif": ("Calculator page 6", "Auto", ""), "bar": ("Calculator page 6", "Auto", "")}
    for k, n, cat in CONSTRUCTION_HEADS: META[k] = (f"Σ {cat} products (page 3)", "Auto", "")
    PCT1 = {"brok", "ovh", "mkt", "cont", "gst"}; PCT2 = {"mkt", "ews", "gst"}
    amounts = RES["costs"].amounts if RES else {}
    tsa = RES["costs"].total_sa if RES else 0
    def shown1(c):
        if c.rate1 is None: return None
        is_pct = c.key in PCT1 or (c.key == "edc" and c.rate2 == "% of construction cost") or \
                 (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
        return c.rate1 * 100 if is_pct else c.rate1
    df = pd.DataFrame([{
        "Cost head": c.name, "How it is calculated": META.get(c.key, ("Spare: per 'Basis'", "per basis", ""))[0],
        "Input 1": shown1(c), "Unit 1": META.get(c.key, ("", "per basis", ""))[1],
        "Input 2": (c.rate2 * 100 if c.key in PCT2 else c.rate2) if isinstance(c.rate2, (int, float)) else None,
        "Unit 2": META.get(c.key, ("", "", ""))[2],
        "Basis": c.rate2 if c.key in ("cont", "edc") else (c.basis if c.key.startswith("spare") else None),
        "Applicable": c.applicable,
        "Phasing curve": ("Follow Collections" if P.info.deal_type == "JV" else P.land.payment_curve) if c.key == "land" else c.curve,
        "Start Q": c.start_q, "End Q": c.end_q,
        "Amount (₹ Cr)": round(amounts.get(c.key, 0.0), 2),
        "₹ / sq ft": round(amounts.get(c.key, 0.0) * CRORE / tsa, 0) if tsa else 0.0,
        "Check": row_status("Costs", c.key)} for c in P.costs])
    ed = st.data_editor(df, key="ed_cost", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Cost head": st.column_config.TextColumn(width="medium"), "How it is calculated": st.column_config.TextColumn(disabled=True),
        "Unit 1": st.column_config.TextColumn(disabled=True), "Unit 2": st.column_config.TextColumn(disabled=True),
        "Basis": st.column_config.SelectboxColumn(options=sorted(set(SPARE_BASES + EDC_BASES + CONTINGENCY_BASES))),
        "Phasing curve": st.column_config.SelectboxColumn(options=COST_CURVES),
        "Start Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "End Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
        "₹ / sq ft": st.column_config.NumberColumn(disabled=True, format="%d"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        auto = {"land", "sd", "aif", "bar"} | {k for k, _, _ in CONSTRUCTION_HEADS}
        for c, (_, r) in zip(P.costs, ed.iterrows()):
            g = lambda col: None if pd.isna(r[col]) else r[col]
            if c.key.startswith("spare"): c.name = str(g("Cost head") or c.name); c.basis = g("Basis")
            c.applicable = bool(r["Applicable"])
            if c.key != "land":
                c.curve = g("Phasing curve") or c.curve
                c.start_q = None if g("Start Q") is None else int(g("Start Q"))
                c.end_q = None if g("End Q") is None else int(g("End Q"))
            if c.key in auto: continue
            v1 = g("Input 1")
            if v1 is not None:
                is_pct = c.key in PCT1 or (c.key == "edc" and (g("Basis") or c.rate2) == "% of construction cost") or \
                         (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
                c.rate1 = float(v1) / 100 if is_pct else float(v1)
            if c.key in ("cont", "edc"):
                c.rate2 = g("Basis") or c.rate2
            elif c.key in PCT2 or c.key == "hotel":
                v2 = g("Input 2")
                c.rate2 = (float(v2) / 100 if c.key in PCT2 else float(v2)) if v2 is not None else c.rate2
        rerun_after_edit("ed_cost")
    if st.button("➕ Add a spare cost head"):
        n = sum(1 for c in P.costs if c.key.startswith("spare")) + 1
        P.costs.append(CostHead(f"spare{n}", f"Other cost {n}", rate1=0.0, applicable=True, curve="Follow Construction",
                                basis="Lump sum (Rs Cr)"))
        rerun_after_edit("ed_cost")
    if RES:
        st.metric("TOTAL COST", f"₹{RES['costs'].total_cost:,.2f} Cr", f"{RES['costs'].total_cost / RES['costs'].total_revenue:.1%} of GDV",
                  delta_color="off")

# ============================================================================ page 6: calculators
elif page.startswith("6"):
    st.header("6 · Calculators")
    t1, t2, t3 = st.tabs(["BAR / premium FAR", "AIF cost", "Initial investment"])
    with t1:
        b = P.bar
        c1, c2 = st.columns(2)
        b.authority = c1.selectbox("Authority", BAR_AUTHORITIES, BAR_AUTHORITIES.index(b.authority))
        b.land_use = c2.selectbox("Land use", BAR_LAND_USES, BAR_LAND_USES.index(b.land_use))
        b.free_far = c1.number_input("Free FAR (× land area)", value=float(b.free_far), min_value=0.0, step=0.1)
        b.loading_factor = c2.number_input("Loading factor (saleable ÷ built-up)", value=float(b.loading_factor), min_value=0.01, step=0.05)
        b.reserved_price_sqm = c1.number_input("Reserved price (₹ / sq m)", value=float(b.reserved_price_sqm), min_value=0.0, step=100.0)
        b.gst_applicable = c2.checkbox("GST on BAR applicable", b.gst_applicable)
        b.gst_rate = pct_in("GST rate on BAR (%)", b.gst_rate)
        st.write("**BAR reserved-price % table** (blank = combination not allowed)")
        df = pd.DataFrame({a: {u: (None if b.pct_table[u].get(a) is None else b.pct_table[u][a] * 100) for u in BAR_LAND_USES}
                           for a in BAR_AUTHORITIES})
        ed = st.data_editor(df, key="ed_bar")
        if not ed.equals(df):
            b.pct_table = {u: {a: (None if pd.isna(ed.loc[u, a]) else float(ed.loc[u, a]) / 100) for a in BAR_AUTHORITIES} for u in BAR_LAND_USES}
            rerun_after_edit("ed_bar")
        show("BAR", "bar")
        if RES:
            x = RES["costs"].bar
            st.success(f"Free {x['free_sqm']:,.0f} sq m · planned {x['planned_sqm']:,.0f} sq m · extra {x['extra_sqm']:,.0f} sq m × "
                       f"₹{b.reserved_price_sqm:,.0f} × {x['pct']:.0%} = **₹{x['cost']:.2f} Cr**")
    with t2:
        a = P.aif
        a.rate = pct_in("Interest rate (% p.a.)", a.rate)
        a.tenure_years = st.number_input("Tenure (years)", value=float(a.tenure_years), min_value=0.0, step=0.5)
        a.compounding = st.selectbox("Compounding", list(COMPOUNDING), list(COMPOUNDING).index(a.compounding))
        show("AIF", "aif")
        if RES:
            x = RES["costs"].aif
            st.info(f"Principal (land + stamp duty + brokerage) ₹{x['principal']:,.2f} Cr → interest **₹{x['cost']:,.2f} Cr**. "
                    f"Switch the 'AIF Cost' head on (page 5) to include it.", icon="💡")
    with t3:
        ii = P.initial
        st.write("**JV – advance to landowner and early payments** (only used in JV deals)")
        c1, c2, c3 = st.columns(3)
        ii.advance_pct = pct_in("Advance to landowner (% of land value at market)", ii.advance_pct)
        ii.dd_amount = c1.number_input("Paid on due diligence (₹ Cr)", value=float(ii.dd_amount), min_value=0.0, step=0.5)
        ii.da_amount = c2.number_input("Paid at DA (₹ Cr)", value=float(ii.da_amount), min_value=0.0, step=0.5)
        show("Initial investment", "dd_da")
        ii.legal_share = pct_in("Share of legal & compliance paid before DA (%)", ii.legal_share)
        ii.bar_share = pct_in("Share of BAR paid upfront (%)", ii.bar_share)
        ii.dev_cost_pre_da = c3.number_input("Development cost before DA (₹ Cr)", value=float(ii.dev_cost_pre_da), min_value=0.0, step=0.5)
        ii.marketing_pre_launch = c3.number_input("Pre-launch marketing (₹ Cr)", value=float(ii.marketing_pre_launch), min_value=0.0, step=0.5)
        labels = {"token": "Term-sheet token (balance)", "dd": "Due diligence", "da": "At DA", "legal": "Legal & compliance",
                  "dev": "Pre-DA development", "marketing": "Pre-launch marketing", "sd": "Stamp duty", "brok": "Brokerage", "bar": "BAR (upfront share)"}
        items = RES["costs"].initial_items if RES else {}
        df = pd.DataFrame([{"Item": labels[k], "Amount (₹ Cr)": round(items.get(k, 0.0), 2), "Payout quarter": q}
                           for k, q in ii.payout_q.items()])
        ed = st.data_editor(df, key="ed_ii", hide_index=True, num_rows="fixed", column_config={
            "Item": st.column_config.TextColumn(disabled=True), "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
            "Payout quarter": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1)})
        if not ed.equals(df):
            for k, (_, r) in zip(list(ii.payout_q), ed.iterrows()):
                ii.payout_q[k] = int(r["Payout quarter"]) if not pd.isna(r["Payout quarter"]) else ii.payout_q[k]
            rerun_after_edit("ed_ii")
        show("Initial investment", "payout_q")
        st.write("**Outright – initial outlay** = land + stamp duty + brokerage + shares below")
        ii.outright_construction_share = pct_in("Share of built-up construction (%)", ii.outright_construction_share)
        ii.outright_legal_share = pct_in("Share of legal & compliance (%)", ii.outright_legal_share)
        if RES:
            cr = RES["costs"]
            st.success(f"Initial investment – JV ₹{cr.initial_jv:,.2f} Cr · Outright ₹{cr.initial_outright:,.2f} Cr · "
                       f"**used ({P.info.deal_type}) ₹{cr.initial_used:,.2f} Cr**")

# ============================================================================ page 7: payment plans
elif page.startswith("7"):
    st.header("7 · Payment plans (CLP)")
    st.caption("Booking-linked: value = quarters after booking (0–12). Construction-linked: value = % of that product's construction "
               "(e.g. 35). Each plan must total exactly 100%. Add as many milestones as you need with ➕.")
    P.catch_up_lag = int(st.number_input("Catch-up lag – late buyers pay milestones already reached this many quarters after booking",
                                         value=int(P.catch_up_lag), min_value=0, max_value=4, step=1))
    show("CLP", "catch_up_lag")
    names = [pl.name for pl in P.plans]
    used_by = {pl: [p.name for p in P.products if p.clp_plan == pl] for pl in names}
    c1, c2 = st.columns([2, 1])
    choice = c1.selectbox("Plan", names, format_func=lambda n: f"{n}  {'⚠️' if issues_for('CLP', n) else '✔'}  "
                                                              f"(used by: {', '.join(used_by[n]) or 'none'})")
    plan = P.plan(choice)
    new_name = c2.text_input("Rename plan", plan.name)
    if new_name and new_name != plan.name and new_name not in names:
        for p in P.products:
            if p.clp_plan == plan.name: p.clp_plan = new_name
        plan.name = new_name
        st.rerun()
    df = pd.DataFrame([{"Milestone": m.name, "Trigger": m.trigger,
                        "Value": m.value if m.trigger == "Booking + quarters" else round(m.value * 100, 4),
                        "% of sale": round(m.pct * 100, 4)} for m in plan.milestones])
    if df.empty:
        df = pd.DataFrame(columns=["Milestone", "Trigger", "Value", "% of sale"])
    ed = st.data_editor(df, key=f"ed_plan_{choice}", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Trigger": st.column_config.SelectboxColumn(options=TRIGGERS, required=True, default="Construction % complete"),
        "Value": st.column_config.NumberColumn(min_value=0, max_value=100, help="quarters (booking) or % (construction)"),
        "% of sale": st.column_config.NumberColumn(min_value=0, max_value=100, step=0.5, format="%.2f")})
    if not ed.equals(df):
        ms = []
        for _, r in ed.iterrows():
            if pd.isna(r["Milestone"]) and pd.isna(r["% of sale"]): continue
            trig = r["Trigger"] if not pd.isna(r["Trigger"]) else "Construction % complete"
            val = 0.0 if pd.isna(r["Value"]) else float(r["Value"])
            ms.append(Milestone(str(r["Milestone"]) if not pd.isna(r["Milestone"]) else "Milestone", trig,
                                val if trig == "Booking + quarters" else val / 100, 0.0 if pd.isna(r["% of sale"]) else float(r["% of sale"]) / 100))
        plan.milestones = ms
        rerun_after_edit(f"ed_plan_{choice}")
    total = sum(m.pct for m in plan.milestones)
    (st.success if abs(total - 1) < 1e-6 else st.error)(f"Plan total: {total:.2%}" + ("" if abs(total - 1) < 1e-6 else " – must be exactly 100%"))
    for i in issues_for("CLP", plan.name):
        if "totals" not in i.message: st.error(i.message, icon="⚠️")
    if RES:
        sched = [s for s in RES["collections"].schedule if s.plan == plan.name and s.due_q]
        if sched:
            st.write("**When construction-linked milestones fall due** (for products using this plan)")
            st.dataframe(pd.DataFrame([{"Product": s.product, "Milestone": s.name, "Falls due": s.text} for s in sched]),
                         hide_index=True, width="stretch")
    st.divider()
    c1, c2, c3 = st.columns(3)
    nn = c1.text_input("New plan name", placeholder="e.g. CLP – 12-Floor Tower")
    if c1.button("➕ Create empty plan") and nn and nn not in names:
        P.plans.append(ClpPlan(nn, [])); st.rerun()
    if c2.button("⧉ Duplicate this plan"):
        k = 2
        while f"{plan.name} ({k})" in names: k += 1
        P.plans.append(ClpPlan(f"{plan.name} ({k})", copy.deepcopy(plan.milestones))); st.rerun()
    if c3.button("🗑 Delete this plan", disabled=bool(used_by[plan.name]),
                 help="Only plans not used by any product can be deleted."):
        P.plans = [pl for pl in P.plans if pl.name != plan.name]; st.rerun()
    show("CLP", "plan_names")

# ============================================================================ page 8: tax & benchmarks
elif page.startswith("8"):
    st.header("8 · Tax, cost of equity & KPI benchmarks")
    f = P.finance
    c1, c2, c3 = st.columns(3)
    with c1:
        f.tax_company = pct_in("Tax rate – Company (%)", f.tax_company, help="Verify with Finance (statutory 115BAA ≈ 25.17%)")
        f.tax_llp = pct_in("Tax rate – LLP (%)", f.tax_llp, help="Verify with Finance (statutory ≈ 34.94%)")
        f.cost_of_equity = pct_in("Cost of equity / discount rate (%)", f.cost_of_equity)
        show("Finance", "finance")
    labels = {"project_irr": "Project IRR (%)", "developer_irr": "Developer IRR (%)", "roce": "ROCE (%)", "cagr": "CAGR (%)",
              "pat_margin": "PAT margin (%)", "ebitda_margin": "EBITDA margin (%)"}
    with c2:
        st.write("**Benchmarks** (green when beaten)")
        for k in ["project_irr", "developer_irr", "roce"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
    with c3:
        st.write("&nbsp;")
        for k in ["cagr", "pat_margin", "ebitda_margin"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
        f.benchmarks["equity_multiple"] = st.number_input("Equity multiple (x)", value=float(f.benchmarks["equity_multiple"]), min_value=0.0, step=0.1)

# ============================================================================ page 9: checks & summary
else:
    st.header("9 · Checks & summary")
    if not issues and RES and not RES["problems"]:
        st.success("✔ ALL CHECKS PASSED – the results below can be used.")
    else:
        for page_name, secs in PAGE_SECTIONS.items():
            mine = [i for i in issues if i.section in secs]
            if mine:
                st.subheader(page_name)
                for i in mine: st.error(i.message, icon="⚠️")
        if RES:
            for p in RES["problems"]:
                if not any(p.endswith(i.message) for i in issues): st.error(p, icon="⚠️")
        if ENGINE_ERROR:
            st.error(f"The engine could not run with these inputs: {ENGINE_ERROR}")
        if RES is None:
            st.info("Results appear once the product, timing and payment-plan errors above are fixed.")
    if RES:
        pnl, m = RES["cashflow"].pnl, RES["cashflow"].metrics
        st.subheader("Headline results")
        c = st.columns(4)
        c[0].metric("GDV", f"₹{pnl['revenue']:,.1f} Cr"); c[1].metric("Total cost", f"₹{pnl['total_cost']:,.1f} Cr")
        c[2].metric("EBITDA", f"₹{pnl['ebitda']:,.1f} Cr", f"{pnl['ebitda_margin']:.1%} margin", delta_color="off")
        c[3].metric("PAT", f"₹{pnl['pat']:,.1f} Cr", f"{pnl['pat_margin']:.1%} margin", delta_color="off")
        c = st.columns(4)
        fmt = lambda v: "n/a" if v is None else f"{v:.1%}"
        c[0].metric("Developer IRR", fmt(m["developer_irr"])); c[1].metric("Developer NPV", f"₹{m['developer_npv']:,.1f} Cr")
        c[2].metric("Peak investment", f"₹{m['peak_investment']:,.1f} Cr", f"in Q{m['peak_quarter']}" if m["peak_quarter"] else "", delta_color="off")
        c[3].metric("Payback", f"Q{m['payback_q']}" if m["payback_q"] else "never",
                    m["payback_date"].strftime("%b %Y") if m["payback_date"] else "", delta_color="off")
        st.caption("Charts, KPI cards and takeaways are on '0 · Dashboard'.")

# ============================================================================ keep results in step with the inputs
# Fields above change the project while this page is drawn; if anything changed, redraw once so every
# number, check and error message reflects the new inputs.
if json.dumps(to_dict(P), sort_keys=True, default=str) != FINGERPRINT:
    st.rerun()


### Cell 6 – New tests (`tests/test_7_stress.py`)

In [ ]:
%%writefile tests/test_7_stress.py
"""STEP 8 – stress testing & Monte Carlo: levers = Excel STRESS sheet, inputs never changed, Monte Carlo reproducible."""
import copy, os

import numpy as np
import pytest

from engine.inputs import to_dict
from engine.stress import McRanges, headroom, monte_carlo, one_way, outcome, scenario_table, stressed, two_way
from conftest import EXCEL_PATH


def test_stressing_never_changes_the_original(project):
    before = to_dict(project)
    stressed(project, price=-0.2, cost=0.3, speed=2, delay=3)
    outcome(project, price=-0.1)
    assert to_dict(project) == before


def test_neutral_levers_give_the_base_case(project, base):
    o = outcome(project)
    assert abs(o["developer_irr"] - base["cashflow"].metrics["developer_irr"]) < 1e-12 and abs(o["pat"] - base["cashflow"].pnl["pat"]) < 1e-9


# golden values from the Excel STRESS sheet
@pytest.mark.parametrize("lever, value, irr, peak, payback", [
    ("price", -0.15, 0.7309871, 33.8055449, 10), ("price", -0.10, 0.8615225333012542, 33.9661, 8), ("price", 0.15, 1.4224, 34.7691, 8),
    ("cost", 0.20, 0.7795, 34.2873, 10), ("speed", 2.0, 0.5062, 35.3701, 16), ("delay", 6, 0.5175, 40.9584, 14)])
def test_one_way_matches_excel(project, lever, value, irr, peak, payback):
    o = outcome(project, **{lever: value})
    assert abs(o["developer_irr"] - irr) < 1e-4 and abs(o["peak"] - peak) < 1e-4 and o["payback_q"] == payback


def test_scenarios_match_excel(project):
    rows = {r["scenario"]: r for r in scenario_table(project)}
    assert abs(rows["Best Case"]["developer_irr"] - 1.5076) < 1e-4 and abs(rows["Best Case"]["gdv"] - 843.15) < 1e-9
    assert abs(rows["Worst Case"]["developer_irr"] - 0.40241041469118866) < 1e-9 and rows["Worst Case"]["payback_q"] == 18


def test_heat_map_corner_matches_excel(project):
    g = two_way(project, "cost", [0.0, 0.20], "price", [-0.10, 0.10], "developer_irr")
    assert abs(g[1, 0] - 0.5085) < 1e-4 and abs(g[0, 1] - 1.3208) < 1e-4


def test_headroom_matches_excel_and_is_consistent(project):
    h = headroom(project)
    assert abs(h["price_to_zero_ebitda"] - (-0.37664608391863835)) < 1e-9
    assert abs(h["cost_to_zero_ebitda"] - 0.6375542865029586) < 1e-9
    assert abs(outcome(project, price=h["price_to_zero_ebitda"])["ebitda"]) < 1e-6          # EBITDA really is zero there
    assert abs(outcome(project, price=h["price_to_hurdle"])["developer_irr"] - 0.18) < 1e-6   # IRR really is the hurdle there


def test_monte_carlo_is_reproducible_and_sensible(project):
    a = monte_carlo(project, n=60, seed=7)
    b = monte_carlo(project, n=60, seed=7)
    assert np.allclose(a["irr"], b["irr"]) and a["p_irr_above_hurdle"] == b["p_irr_above_hurdle"]
    assert 0 <= a["p_irr_above_hurdle"] <= 1 and a["irr_p10"] <= a["irr_p50"] <= a["irr_p90"]
    assert a["failed_runs"] == 0


def test_monte_carlo_with_no_uncertainty_equals_base(project, base):
    r = McRanges((0, 0, 0), (0, 0, 0), (1, 1, 1), {0: 1.0})
    mc = monte_carlo(project, r, n=10)
    assert np.allclose(mc["irr"], base["cashflow"].metrics["developer_irr"])


def test_harsher_ranges_lower_the_probability(project):
    soft = monte_carlo(project, n=80, seed=1)
    harsh = monte_carlo(project, McRanges((-0.45, -0.30, -0.15), (0.20, 0.35, 0.50), (1.5, 2.0, 2.5), {2: 0.5, 3: 0.5}), n=80, seed=1)
    assert harsh["irr_p50"] < soft["irr_p50"] and harsh["p_irr_above_hurdle"] <= soft["p_irr_above_hurdle"]


@pytest.mark.skipif(not os.path.exists(EXCEL_PATH), reason="Excel FINAL file not found in data/")
def test_full_stress_sheet_matches_excel(project):
    from engine.excel_check import read_excel_stress
    x = read_excel_stress(EXCEL_PATH)
    for sec, lever in [("C1", "price"), ("C2", "cost"), ("C3", "speed"), ("C4", "delay")]:
        for py, ex in zip(one_way(project, lever, [r["value"] for r in x[sec]]), x[sec]):
            assert abs(py["developer_irr"] - ex["Developer IRR"]) < 1e-9 and abs(py["peak"] - ex["Peak investment (₹ Cr)"]) < 1e-9
            assert f"Q{py['payback_q']}" == ex["Payback"]
    for sec, (rl, cl, met) in {"D1": ("cost", "price", "developer_irr"), "D2": ("cost", "price", "pat"), "D3": ("speed", "delay", "peak")}.items():
        d = x[sec]
        assert np.abs(two_way(project, rl, d["row_values"], cl, d["col_values"], met) - d["grid"]).max() < 1e-9


def test_stress_charts_build(project):
    pytest.importorskip("plotly")
    import sys
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    import stress_view as sv
    T = sv.compute_tables(project)
    mc = monte_carlo(project, n=30)
    figs = [sv.fig_one_way(T["one_way"]["price"], "price", 0.18), sv.fig_tornado(T["tornado"]),
            sv.fig_heat(T["irr_grid"], [0, .05, .1, .15, .2], [-.1, -.05, 0, .05, .1], "{:+.0%}", "{:+.0%}", "t", "{:.0%}"),
            sv.fig_mc_hist(mc, 0.18), sv.fig_mc_exceedance(mc, 0.18), sv.fig_mc_scatter(mc)]
    assert all(len(f.data) > 0 for f in figs)


### Cell 7 – Safety net: run all tests (expect ✅ 165, takes ~15 seconds)

In [ ]:
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--tb=short'], capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1])
print('✅ All good – continue' if res.returncode == 0 else '❌ Something broke:\n' + res.stdout[-4000:])

### Cell 8 – Preview the stress results here in Colab
Scenario comparison, tornado, heat map, headroom – and two Monte Carlo runs: your **default ranges** and a **deliberately harsh** set,
to show how the probability falls when the market turns against you.

In [ ]:
import importlib, pandas as pd, numpy as np
import engine.stress, stress_view
importlib.reload(engine.stress); importlib.reload(stress_view)
from engine.inputs import sample_project
from engine.stress import McRanges, monte_carlo
import stress_view as sv

project = sample_project()
T = sv.compute_tables(project)
df = pd.DataFrame(T['scenarios'])[['scenario', 'gdv', 'pat', 'developer_irr', 'developer_npv', 'peak', 'payback_q']]
print('SCENARIO COMPARISON'); display(df.round(3))
h = T['headroom']
print(f"\nHEADROOM: prices can fall {-h['price_to_hurdle']:.1%} before developer IRR hits the {h['hurdle']:.0%} hurdle "
      f"(and {-h['price_to_zero_ebitda']:.1%} before EBITDA is zero); construction can overrun {h['cost_to_hurdle']:.1%} before IRR hits the hurdle.")
sv.fig_tornado(T['tornado']).show()
sv.fig_heat(T['irr_grid'], [0, .05, .1, .15, .2], [-.1, -.05, 0, .05, .1], '{:+.0%}', '{:+.0%}', 'Developer IRR: price × cost', '{:.0%}',
            y_title='cost', x_title='price').show()

for label, ranges in [('DEFAULT ranges', McRanges()),
                      ('DELIBERATELY HARSH ranges', McRanges((-0.40, -0.20, 0.0), (0.05, 0.25, 0.50), (1.0, 1.6, 2.4), {0: .2, 1: .3, 2: .3, 3: .2}))]:
    t0 = time.time(); mc = monte_carlo(project, ranges, n=1000, seed=42)
    print(f"\nMONTE CARLO – {label} (1,000 runs in {time.time() - t0:.0f} s)")
    print(f"  P(developer IRR ≥ {mc['hurdle']:.0%}) = {mc['p_irr_above_hurdle']:.1%} | P(NPV > 0) = {mc['p_npv_positive']:.1%} | P(loss) = {mc['p_loss']:.1%}")
    print(f"  IRR P10 / P50 / P90 = {mc['irr_p10']:.0%} / {mc['irr_p50']:.0%} / {mc['irr_p90']:.0%} | peak investment P90 ₹{mc['peak_p90']:.1f} Cr")
    sv.fig_mc_exceedance(mc, mc['hurdle']).show()

### Cell 9 – ▶ Start the app (Option A – Google link)

In [ ]:
subprocess.run('pkill -f "streamlit run"', shell=True); time.sleep(1)
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true',
                  '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
                 cwd=BASE, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
alive = False
for _ in range(30):
    time.sleep(1)
    try:
        alive = urllib.request.urlopen('http://localhost:8501/_stcore/health', timeout=2).read() == b'ok'
        if alive: break
    except Exception:
        pass
print('✅ App running' if alive else '❌ App did not start – log:\n' + open('/content/streamlit.log').read()[-2000:])
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8501, anchor_text='👉 Click here to open the Feasibility app')
except ImportError:
    print('Open http://localhost:8501')

**Option B** – public link (other devices; some office networks block it).

In [ ]:
import re
subprocess.run('pkill -f cloudflared', shell=True)
CF = '/content/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF); os.chmod(CF, 0o755)
subprocess.Popen([CF, 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
print('Open: ' + url if url else 'No link – use Option A.')

### Cell 10 – What to try on **S · Stress & Monte Carlo**
1. **A · Live what-if** – drag *Sales price shock* to −10% → developer IRR 110.2% → **86.2%** (same as the Excel table). Add *Extra delay* 8 → **36.7%**; the red line shows the deeper, longer funding curve.
2. **B · Tornado** – sales speed and delay move IRR more than price or cost: cash timing matters most in a pre-sales funded project.
3. **D · Heat maps** – find the red corner where price falls and costs rise together.
4. **E · Headroom** – prices can fall **32.8%** before IRR reaches the 18% hurdle; costs can overrun **53.6%**.
5. **F · Monte Carlo** – press ▶ Run (1,000 runs ≈ 10 s). With the default ranges the sample deal clears 18% in every run, so move the
   *probability slider* to e.g. 60% → about **65%** chance. Then try harsher ranges (e.g. price −40 / −20 / 0, cost 5 / 25 / 50) and re-run.

### Cell 11 – Stop the app

In [ ]:
subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True); print('Stopped.')

---
### ✅ Step 8 complete
**Next – Step 9:** save / load / export – a downloadable **Excel report** (inputs, P&L, cash flow, key metrics, stress results) and a
PDF-ready summary, so the app's results can be shared with anyone.